# Conclusion — Rapport à la direction (synthèse)

**Contexte métier.** Inved Corp veut un **outil assistif d'estimation** pour ses consultants : sur soumission d'un formulaire, l'outil renvoie un **prix estimé + intervalle de confiance + top-3 facteurs explicatifs**, le consultant gardant la **validation finale**. Deux usages : le **conseil rénovation** (quand l'estimation IA < attente client, on montre les leviers actionnables, ex. `KitchenQual`) et le label **« Prix certifié par Inved AI »** (quand IA ≈ marché). KPIs visés : temps d'expertise **4 h → 2 h**, **+20 %** de ventes sous 90 jours, **< 10 %** d'override manuel.

**Modèle final (verdict Phase 5, non rejugé ici).** Champion = **Stacking** (Lasso + GradientBoosting + XGBoost → méta-RidgeCV), défendu sur les trois axes :
- **Mathématique** — RMSLE holdout **0,1122** (OOF 0,111), mais en **quasi ex-æquo statistique** (IC bootstrap chevauchant, NB4 §5.2.1) ; sa **diversité d'erreurs** fait la différence.
- **Informatique** — latence **~56 ms/requête** (≈ 90× sous le SLA de 5 s) ; son vrai coût est la **surface de monitoring** (4 sous-modèles), pas le calcul.
- **Métier** — interprétabilité indirecte (proxy SHAP) ; **équité** correcte (résidus par quartier dans ≈ [−4 %, +6 %]).

**Recommandation.** Livrer le **Stacking** pour le label « certifié ». **Repli légitime** : **CatBoost** ou **XGBoost tuné** (Δ ≈ 0,005 RMSLE, dans le bruit), **1 seul artefact** à servir/monitorer, si la direction privilégie la simplicité d'exploitation.

**Déploiement (résumé).** API **temps réel** (réponse < 5 s sur soumission), **MLflow Model Registry → `mlflow models serve`** comme chemin d'inférence canonique, **réentraînement** déclenché en OU — **cadence mensuelle** (fenêtre glissante) · **RMSLE réalisée > 0,13** · **PSI > 0,25** sur une variable clé (cf. §7.4).

**Risques majeurs.** Opacité du Stacking, **dérive du marché / gentrification**, équité sur les quartiers à faible volume. Détail ci-dessous.

## Récapitulatif du parcours CRISP-ML(Q)

| Phase | Livrable clé |
|---|---|
| 1 — Compréhension métier | ML Canvas : outil assistif d'estimation, 2 usages (conseil rénovation, label « certifié »), KPIs (4 h→2 h, +20 % ventes, < 10 % override). |
| 2 — Compréhension des données | EDA Ames (1460 biens, 79 variables), cible log-normale → **RMSLE**, tests Shapiro-Wilk + ANOVA, audit de cardinalité. |
| 3 — Préparation | Anti-fuite (imputation dans le Pipeline), feature engineering (8 variables dérivées), encodage hybride (Ordinal / TargetEncoder / OHE), 3 préprocesseurs. |
| 4 — Modélisation | 17 modèles, 4 familles (linéaires/KNN/MLP · arbres sklearn · boosting natif · Stacking), tuning Optuna. |
| 5 — Évaluation | 3 axes (math/système/métier), IC bootstrap (quasi ex-æquo), champion **Stacking 0,1122**, seuil 0,13, signoff équité. |
| 6 — Déploiement | Refit 100 %, soumission Kaggle, registre + service MLflow, stratégie CI/CD, coût, PoC (bonus). |
| 7 — Surveillance (CRISP-ML(Q)) | 3 couches de monitoring, dérive PSI, réentraînement, rollout A/B, équité continue. |

## Le modèle final en chiffres

- **Champion** : Stacking (Lasso + GradientBoosting + XGBoost → méta-RidgeCV).
- **Performance** : RMSLE holdout **0,1122** (OOF 0,1108), IC95 bootstrap ≈ [0,096 – 0,130].
- **Latence** : ~56 ms/requête (≈ 90× sous le SLA de 5 s).
- **Équité** : résidus OOF par quartier dans ≈ [−4 %, +6 %].
- **Repli chiffré** : CatBoost / XGBoost tuné (Δ ≈ 0,005 RMSLE, ¼ de la surface de monitoring).

## Risques et limites du champion

- **Opacité du Stacking.** Le modèle servi n'a **pas d'attribution directe** : le « top-3 facteurs » promis au consultant est calculé via **SHAP sur XGBoost** comme proxy lisible (§4.3.7), donc une **approximation** du raisonnement de l'ensemble — à présenter comme telle, et à valider humainement.
- **Pas d'extrapolation hors domaine.** Modèles à base d'arbres : un bien plus grand ou plus cher que tout l'historique voit sa prédiction **plafonnée** (cf. les plus grosses erreurs, NB4 §5.5.2.2). Les biens d'exception restent du ressort du consultant.
- **Équité sur quartiers à faible volume.** Le léger penchant à sous-estimer BrkSide / IDOTRR (~+5–6 %, NB4 §5.5.4) est un **signal mineur à suivre**, pas un biais systémique — mais il doit rester sous surveillance (couches 1 + 3).
- **Dérive du marché / gentrification.** Cf. §7.5 — le risque structurel principal, traité par la fenêtre glissante mensuelle + déclencheur de réentraînement.
- **Quasi ex-æquo (rappel).** Le Stacking n'est **pas significativement meilleur** que CatBoost / XGBoost tuné (IC chevauchants) ; un **repli vers un modèle unique** simplifierait l'exploitation (¼ de la surface de monitoring) pour une perte de RMSLE dans le bruit. Décision réversible, à réévaluer en production.

## Perspectives

- **PoC applicatif** (bonus) : interface NiceGUI cliente de l'API MLflow (§6.6 / dossier `app/`).
- **Mise en production réelle** : héberger `mlflow models serve` sur une instance cloud CPU — l'architecture localhost du PoC s'y transpose 1:1 (cf. ML Canvas).
- **Boucle d'amélioration** : réentraînement mensuel + suivi de dérive (Phase 7) alimentant de nouvelles versions du registre.
- **Au-delà** : enrichissement par données externes (dérive de prix de marché), arbitrage vers un modèle unique si l'exploitation prime sur les 0,005 de RMSLE.

---

*Fin du livrable. Du besoin métier d'Inved Corp au modèle servi et surveillé : le parcours **CRISP-ML(Q)** complet — de la compréhension métier à la surveillance & maintenance en production — chaque phase autonome et justifiée.*